# Dashboards do Site

Painéis finais de Chamados e Superadmin, gerados a partir das tabelas Gold.

Execute em um compute Python do Databricks.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# Filtros interativos do dashboard no Databricks
dbutils.widgets.dropdown("filtro_status", "TODOS", ["TODOS", "REGISTRADO", "EM_ATENDIMENTO", "ATENDIDO"])
dbutils.widgets.dropdown("filtro_categoria", "TODOS", ["TODOS", "DUVIDA", "SUGESTAO", "RECLAMACAO", "OUTRO"])
dbutils.widgets.text("filtro_responsavel", "")
dbutils.widgets.text("filtro_data_inicio", "")
dbutils.widgets.text("filtro_data_fim", "")
dbutils.widgets.dropdown("filtro_nivel_acesso", "TODOS", ["TODOS", "OPERADOR", "GESTOR", "ADMIN", "SUPER_ADMIN"])

FILTRO_STATUS = dbutils.widgets.get("filtro_status").strip()
FILTRO_CATEGORIA = dbutils.widgets.get("filtro_categoria").strip()
FILTRO_RESPONSAVEL = dbutils.widgets.get("filtro_responsavel").strip()
FILTRO_DATA_INICIO = dbutils.widgets.get("filtro_data_inicio").strip()
FILTRO_DATA_FIM = dbutils.widgets.get("filtro_data_fim").strip()
FILTRO_NIVEL_ACESSO = dbutils.widgets.get("filtro_nivel_acesso").strip()

chamados = spark.table("gold_chamados").toPandas()
usuarios = spark.table("gold_superadmin_usuarios").toPandas()
rpa = spark.table("gold_superadmin_rpa").toPandas()

chamados["data_hora_abertura"] = pd.to_datetime(chamados["data_hora_abertura"])
chamados["data_hora_encerramento"] = pd.to_datetime(chamados["data_hora_encerramento"])

def data_filtro(valor, nome):
    if not valor:
        return None
    data = pd.to_datetime(valor, errors="coerce")
    if pd.isna(data):
        raise ValueError(f"{nome} deve estar no formato AAAA-MM-DD.")
    return data

data_inicio = data_filtro(FILTRO_DATA_INICIO, "filtro_data_inicio")
data_fim = data_filtro(FILTRO_DATA_FIM, "filtro_data_fim")
if data_inicio is not None and data_fim is not None and data_inicio > data_fim:
    raise ValueError("filtro_data_inicio não pode ser posterior ao filtro_data_fim.")

if FILTRO_STATUS != "TODOS":
    chamados = chamados[chamados["status"] == FILTRO_STATUS]
if FILTRO_CATEGORIA != "TODOS":
    chamados = chamados[chamados["categoria"] == FILTRO_CATEGORIA]
if FILTRO_RESPONSAVEL:
    chamados = chamados[chamados["cod_usuario_responsavel"].astype("string") == FILTRO_RESPONSAVEL]
if data_inicio is not None:
    chamados = chamados[chamados["data_hora_abertura"] >= data_inicio]
if data_fim is not None:
    chamados = chamados[chamados["data_hora_abertura"] < data_fim + pd.Timedelta(days=1)]
if FILTRO_NIVEL_ACESSO != "TODOS":
    usuarios = usuarios[usuarios["nivel_acesso"] == FILTRO_NIVEL_ACESSO]

PALETA = {
    "fundo": "#F8FAFC",
    "texto": "#0F172A",
    "secundario": "#64748B",
    "azul": "#2563EB",
    "verde": "#16A34A",
    "laranja": "#EA580C",
    "roxo": "#7C3AED",
    "vermelho": "#DC2626",
    "ciano": "#0891B2",
}

def preparar_eixo(ax, titulo, xlabel=None, ylabel=None):
    ax.set_title(titulo, loc="left", fontsize=13, fontweight="bold", color=PALETA["texto"], pad=12)
    ax.set_facecolor("white")
    ax.grid(axis="y", color="#E2E8F0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#CBD5E1")
    ax.tick_params(colors=PALETA["secundario"], labelsize=9)
    if xlabel: ax.set_xlabel(xlabel, color=PALETA["secundario"])
    if ylabel: ax.set_ylabel(ylabel, color=PALETA["secundario"])

def card(fig, x, label, value, color):
    fig.text(x, 0.91, label.upper(), fontsize=9, color=PALETA["secundario"], weight="bold")
    fig.text(x, 0.855, value, fontsize=22, color=color, weight="bold")

def valor_tempo(valor):
    return "—" if pd.isna(valor) else f"{valor:,.1f} min".replace(",", "X").replace(".", ",").replace("X", ".")


In [ ]:
# Painel de Chamados
total = len(chamados)
abertos = int(chamados["esta_aberto"].sum())
finalizados = int(chamados["esta_finalizado"].sum())
tempo_medio = chamados.loc[chamados["esta_finalizado"], "tempo_resolucao_minutos"].mean()

fig = plt.figure(figsize=(18, 11), facecolor=PALETA["fundo"])
fig.suptitle("Skadi · Dashboard de Chamados", x=0.055, y=0.975, ha="left", fontsize=22, fontweight="bold", color=PALETA["texto"])
fig.text(0.055, 0.945, "Visão operacional dos chamados registrados no sistema", color=PALETA["secundario"], fontsize=11)
card(fig, 0.055, "Total de chamados", f"{total:,}", PALETA["azul"])
card(fig, 0.28, "Em aberto", f"{abertos:,}", PALETA["laranja"])
card(fig, 0.50, "Finalizados", f"{finalizados:,}", PALETA["verde"])
card(fig, 0.72, "Tempo médio", valor_tempo(tempo_medio), PALETA["roxo"])
gs = gridspec.GridSpec(2, 4, figure=fig, left=0.055, right=0.97, top=0.78, bottom=0.09, hspace=0.45, wspace=0.30)
ax_status = fig.add_subplot(gs[0, 0])
ax_categoria = fig.add_subplot(gs[0, 1])
ax_responsavel = fig.add_subplot(gs[0, 2:])
ax_evolucao = fig.add_subplot(gs[1, :2])
ax_hist = fig.add_subplot(gs[1, 2])
ax_box = fig.add_subplot(gs[1, 3])

status = chamados["status"].value_counts().sort_values()
status.plot.barh(ax=ax_status, color=PALETA["azul"], width=0.65)
preparar_eixo(ax_status, "Chamados por status", ylabel="Status")

categoria = chamados["categoria"].value_counts().sort_values()
categoria.plot.barh(ax=ax_categoria, color=PALETA["verde"], width=0.65)
preparar_eixo(ax_categoria, "Chamados por categoria", ylabel="Categoria")

responsavel = chamados["cod_usuario_responsavel"].fillna("NAO ATRIBUIDO").astype(str).value_counts().sort_values()
responsavel.plot.barh(ax=ax_responsavel, color=PALETA["roxo"], width=0.65)
preparar_eixo(ax_responsavel, "Chamados por responsável", xlabel="Quantidade", ylabel="Código do responsável")

abertura = chamados.assign(dia=chamados["data_hora_abertura"].dt.date).groupby("dia").size().rename("Abertos")
encerramento = chamados.loc[chamados["esta_finalizado"]].assign(dia=lambda x: x["data_hora_encerramento"].dt.date).groupby("dia").size().rename("Finalizados")
evolucao = pd.concat([abertura, encerramento], axis=1).fillna(0).sort_index()
evolucao.plot(ax=ax_evolucao, marker="o", linewidth=2, color=[PALETA["azul"], PALETA["verde"]])
preparar_eixo(ax_evolucao, "Evolução diária", xlabel="Dia", ylabel="Quantidade")
ax_evolucao.legend(frameon=False)
ax_evolucao.tick_params(axis="x", rotation=45)

tempos = chamados.loc[chamados["esta_finalizado"], "tempo_resolucao_minutos"].dropna()
ax_hist.hist(tempos, bins=15, color=PALETA["laranja"], edgecolor="white")
preparar_eixo(ax_hist, "Distribuição do tempo", xlabel="Minutos", ylabel="Quantidade")
ax_box.boxplot(tempos, vert=True, patch_artist=True, boxprops={"facecolor": PALETA["roxo"], "alpha": 0.65}, medianprops={"color": PALETA["texto"], "linewidth": 2})
preparar_eixo(ax_box, "Boxplot do tempo", ylabel="Minutos")
ax_box.set_xticks([])
display(fig)
plt.close(fig)

In [ ]:
# Painel de Superadmin
total_usuarios = int(usuarios["quantidade_usuarios"].sum()) if not usuarios.empty else 0
ultima_rpa = rpa["ultima_execucao_sucesso"].max() if not rpa.empty else None

fig = plt.figure(figsize=(18, 10), facecolor=PALETA["fundo"])
fig.suptitle("Skadi · Dashboard de Superadmin", x=0.055, y=0.975, ha="left", fontsize=22, fontweight="bold", color=PALETA["texto"])
fig.text(0.055, 0.945, "Visão gerencial de usuários, chamados e execução do RPA", color=PALETA["secundario"], fontsize=11)
card(fig, 0.055, "Total de usuários", f"{total_usuarios:,}", PALETA["ciano"])
card(fig, 0.28, "Chamados abertos", f"{abertos:,}", PALETA["laranja"])
card(fig, 0.50, "Tempo médio", valor_tempo(tempo_medio), PALETA["roxo"])
card(fig, 0.72, "Última carga RPA", str(ultima_rpa) if pd.notna(ultima_rpa) else "—", PALETA["verde"])
gs = gridspec.GridSpec(2, 3, figure=fig, left=0.055, right=0.97, top=0.78, bottom=0.10, hspace=0.45, wspace=0.30)
ax_perfil = fig.add_subplot(gs[0, 0])
ax_status = fig.add_subplot(gs[0, 1])
ax_evolucao = fig.add_subplot(gs[0, 2])
ax_hist = fig.add_subplot(gs[1, 0:2])
ax_box = fig.add_subplot(gs[1, 2])

usuarios_por_perfil = usuarios.set_index("nivel_acesso")["quantidade_usuarios"].sort_values()
usuarios_por_perfil.plot.barh(ax=ax_perfil, color=PALETA["ciano"], width=0.65)
preparar_eixo(ax_perfil, "Usuários por nível de acesso", xlabel="Quantidade", ylabel="Perfil")

status = chamados["status"].value_counts().sort_values()
status.plot.barh(ax=ax_status, color=PALETA["vermelho"], width=0.65)
preparar_eixo(ax_status, "Chamados por status", xlabel="Quantidade", ylabel="Status")

evolucao.plot(ax=ax_evolucao, marker="o", linewidth=2, color=[PALETA["azul"], PALETA["verde"]])
preparar_eixo(ax_evolucao, "Evolução diária", xlabel="Dia", ylabel="Quantidade")
ax_evolucao.legend(frameon=False, fontsize=8)
ax_evolucao.tick_params(axis="x", rotation=45)

ax_hist.hist(tempos, bins=15, color=PALETA["laranja"], edgecolor="white")
preparar_eixo(ax_hist, "Distribuição do tempo de resolução", xlabel="Minutos", ylabel="Quantidade")
ax_box.boxplot(tempos, vert=True, patch_artist=True, boxprops={"facecolor": PALETA["roxo"], "alpha": 0.65}, medianprops={"color": PALETA["texto"], "linewidth": 2})
preparar_eixo(ax_box, "Boxplot do tempo", ylabel="Minutos")
ax_box.set_xticks([])
display(fig)
plt.close(fig)